In [2]:
!pip install pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 3.3 MB/s eta 0:00:00a 0:00:01


In [3]:
!pip install mistralai

  Using cached eval_type_backport-0.2.2-py3-none-any.whl.metadata (2.2 kB)
  Using cached jsonpath_python-1.0.6-py3-none-any.whl.metadata (12 kB)
  Using cached typing_inspect-0.9.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.27.2-cp310-cp310-macosx_11_0_arm64.whl.metadata (6.6 kB)
Using cached eval_type_backport-0.2.2-py3-none-any.whl (5.8 kB)
Using cached jsonpath_python-1.0.6-py3-none-any.whl (7.6 kB)
Using cached pydantic_core-2.27.2-cp310-cp310-macosx_11_0_arm64.whl (1.8 MB)
Using cached typing_inspect-0.9.0-py3-none-any.whl (8.8 kB)
Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)
Using cached h11-0.14.0-py3-none-any.whl (58 kB)


In [5]:
!pip install requests

  Using cached requests-2.32.3-py3-none-any.whl.metadata (4.6 kB)
Using cached requests-2.32.3-py3-none-any.whl (64 kB)


In [1]:
from PIL import Image

import base64
import os
import requests

import pandas as pd
import numpy as np
import time

from pydantic import BaseModel

from mistralai import Mistral
from utils import read_red_channel, read_green_channel, read_blue_channel, generate_random_sample

In [2]:
os.environ["PIXTRAL"] = "heS2sEW6opewKusGCOt6wy4stouOpwEN"

In [3]:
# Helper function
def encode_image(image_path):
    """Encode the image to base64."""
    try:
        with open(image_path, "rb") as image_file:
            return base64.b64encode(image_file.read()).decode('utf-8')
    except FileNotFoundError:
        print(f"Error: The file {image_path} was not found.")
        return None
    except Exception as e:  # Added general exception handling
        print(f"Error: {e}")
        return None

# Specify model
model = "pixtral-12b-2409"

# Path to your image
image_path = "data/converted/19.jpg"

# Getting the base64 string
base64_image = encode_image(image_path)

api_key = os.environ["PIXTRAL"]
client = Mistral(api_key=api_key)

## Demo

In [74]:
# Demo
demo_img = '/Users/mohit/Documents/GitHub/ecdna-analysis/test_im_data/labels/1214.png'
# Encode image
img_input = encode_image(demo_img)

# Define the messages for the chat
messages = [
    {
        "role":"system",
        "content": "You are a pathologist analyzing metaphase cell images. These images contain three main structures: ecDNA, nuclei, and chromosomes. Your goal is to focus on connected component analysis in the image."
        
    },
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": "How many ecDNA are in the image? Return an integer number."
            },
            {
                "type": "image_url",
                "image_url": f"data:image/jpeg;base64,{img_input}" 
            }
        ]
    }
]

# Get the chat response
chat_response = client.chat.complete(
    model=model,
    temperature=0.7,
    messages=messages
)

# Print the content of the response
print(chat_response.choices[0].message.content)

To determine the number of ecDNA (extrachromosomal DNA) in the image, we need to perform a connected component analysis. This involves identifying distinct, connected regions in the image that represent the ecDNA structures.

Here's a step-by-step approach to analyze the image:

1. **Convert the Image to Binary**: Convert the image to a binary format where ecDNA regions are represented by white pixels and the background is represented by black pixels.

2. **Label Connected Components**: Use a connected component labeling algorithm to identify distinct regions in the binary image. Each connected component will correspond to a single ecDNA structure.

3. **Count the Components**: Count the number of distinct connected components identified in the previous step.

Assuming the image is already in a binary format with ecDNA regions in white and the background in black, we can proceed with the labeling and counting:

- **Connected Component Labeling**: This can be done using various algorith

### Generate and load data

In [ ]:
# Helper functions to generate metrics

def filter_data(actual, predicted, threshold = 1000):
    mask_0 = actual != 0
    actual = actual[mask_0]
    predicted = predicted[mask_0]

    threshold_mask = predicted < threshold
    actual = actual[threshold_mask]
    predicted = predicted[threshold_mask]
    
    return actual, predicted
    
def calc_metrics(actual, predicted):
    actual, predicted = filter_data(actual, predicted)

    RMSE = np.sqrt(((predicted - actual) ** 2).mean())
    MAE = (((predicted - actual)).abs()).mean()
    MAPE = (((predicted - actual) / actual).abs()).mean()
    
    return MAE, RMSE, MAPE

In [5]:
# Load train data into a df
HOME = '/Users/mohit/Documents/GitHub/ecdna-analysis'
train_df = pd.read_csv(HOME + "/train_im_data" + '/train_ec_quantification.csv', header=0, names=['img','ec'])
train_df['img'] = train_df['img'].apply(lambda x: x[:-4])
train_df.head()

,img,ec
0,2343,0
1,2344,0
2,2345,0
3,2346,0
4,2347,1


In [6]:
# Load the test data
test_df = pd.read_csv(HOME + "/test_im_data" + '/test_im_ec_quantification.csv', header=0, names=['img','ec'])
test_df['img'] = test_df['img'].apply(lambda x: x[:-4])
test_df.head()

,img,ec
0,0,1
1,1,3
2,10,2
3,1000,2
4,1001,0


In [7]:
def generate_response(client, model, img_path, prompt, context, response_format=None, temp=0.7):
    
    # Encode image
    img_input = encode_image(img_path)
    
    # Define the messages for the chat
    messages = [
        {
            "role":"system",
            "content": f'{context}'
            
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": f"{prompt}"
                },
                {
                    "type": "image_url",
                    "image_url": f"data:image/jpeg;base64,{img_input}" 
                }
            ]
        }
    ]
    
    # Get the chat response
    chat_response = client.chat.complete(
        model=model,
        temperature=temp,
        messages=messages
    )

    # Return the content of the response
    return chat_response.choices[0].message.content

### Experiment #1: 0-Shot Learning

In [8]:
def pixtral_zero_shot(data_folder):
    ec_predictions = []
    context = """
            For this task, you will act as a pathologist who is studying 
            extrachromosomal DNA (ecDNA). You will be given multiple images
            and asked to identify the number of circular ecDNA structures.
            ecDNA is usually smaller than chromosomes or nuclei which are also present.
            """
    prompt = "Count the number of ecDNA in this image. Output only a single integer and no additional text."
    
    model = "pixtral-12b-2409"
    api_key = os.environ["PIXTRAL"]
    client = Mistral(api_key=api_key)
    img_list = os.listdir(data_folder)
    
    for img in img_list:
        if img.endswith('.png'):    
            print(img)
            
            img_path = os.path.join(data_folder, img)
            result = generate_response(client, model, img_path, prompt, context)
            ec_predictions.append(result)
            
            time.sleep(3)
        
    preds = pd.DataFrame(data={'img':img_list, 'pred':ec_predictions})
    preds['img'] = preds['img'].apply(lambda x: x[:-4])
    return preds

In [64]:
test_folder = '/Users/mohit/Documents/GitHub/ecdna-analysis/test_im_data/labels'
sampled_path = './data/sampled'
sample_df = generate_random_sample(input_folder=test_folder, output_folder=sampled_path, sample_size=100)

preds = pixtral_zero_shot(sampled_path)


1179.png
77.png
228.png
1030.png
1554.png
439.png
411.png
1225.png
174.png
2328.png
2310.png
1235.png
1779.png
2264.png
207.png
993.png
70.png
830.png
617.png
577.png
614.png
1873.png
854.png
14.png
263.png
2214.png
504.png
909.png
712.png
29.png
2189.png
658.png
894.png
710.png
1253.png
1086.png
1050.png
1722.png
2149.png
498.png
517.png
1901.png
1645.png
892.png
1453.png
845.png
500.png
1297.png
703.png
1661.png
2022.png
2157.png
2194.png
492.png
479.png
2009.png
255.png
297.png
1059.png
446.png
2233.png
1706.png
251.png
469.png
2153.png
870.png
2231.png
1895.png
1117.png
497.png
1825.png
1831.png
2323.png
157.png
1213.png
2322.png
354.png
626.png
1013.png
1990.png
2296.png
2335.png
2321.png
145.png
50.png
2251.png
2046.png
193.png
839.png
1605.png
434.png
350.png
191.png
2118.png
595.png
1558.png
2045.png
85.png
1174.png
1809.png


In [66]:
baseline_df= pd.merge(preds, test_df, how='left', on='img')
baseline_df

,img,pred,ec
0,1179,1,0
1,77,5,11
2,228,5,7
3,1030,13,2
4,1554,15,32
...,...,...,...
95,1558,1,6
96,2045,1,0
97,85,5,11
98,1174,11,27


In [84]:
# Calculate the metrics
calc_metrics(baseline_df['ec'].astype(int), baseline_df['pred'].astype(int))

(np.float64(10.710144927536232),
 np.float64(18.63960253354925),
 np.float64(0.6918278554902101))

### Experiment #2: 3-shot Learning

In [9]:
def generate_three_shot_response(client, model, three_shot_imgs, input_img_path, prompt, context, temp=0.7):
    
    # Encode image
    img_input = encode_image(input_img_path)
    
    # Define the messages for the chat
    messages = [
        {
            "role":"system",
            "content": f'{context}'
            
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Here is the first image which has 1 ecDNA."
                },
                {
                    "type": "image_url",
                    "image_url": f"data:image/jpeg;base64,{three_shot_imgs[0]}" 
                }
            ]
        },
        {
            "role": "assistant",
            "content": [
                {
                    "type": "text",
                    "text": "Understood."
                }
            ]
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Here is the second image which has 19 ecDNA."
                },
                {
                    "type": "image_url",
                    "image_url": f"data:image/jpeg;base64,{three_shot_imgs[1]}" 
                }
            ]
        },
        {
            "role": "assistant",
            "content": [
                {
                    "type": "text",
                    "text": "Understood."
                }
            ]
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Here is the first image which has 129 ecDNA."
                },
                {
                    "type": "image_url",
                    "image_url": f"data:image/jpeg;base64,{three_shot_imgs[2]}" 
                }
            ]
        },
        {
            "role": "assistant",
            "content": [
                {
                    "type": "text",
                    "text": "Got it, I am ready for the final input image."
                }
            ]
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": f"{prompt}"
                },
                {
                    "type": "image_url",
                    "image_url": f"data:image/jpeg;base64,{img_input}" 
                }
            ]
        }
    ]
    
    # Get the chat response
    chat_response = client.chat.complete(
        model=model,
        temperature=temp,
        messages=messages
    )

    # Return the content of the response
    return chat_response.choices[0].message.content

In [10]:
def pixtral_three_shot(data_folder):
    ec_predictions = []
    context = """
                For this task, you will act as a pathologist who is studying 
                extrachromosomal DN A (ecDNA). You will be given multiple images
                and asked to identify the number of circular ecDNA structures.
                ecDNA is usually smaller than chromosomes or nuclei which are also present.
                I will first provide 3 example images and counts.
            """
    prompt = "Count the number of ecDNA in this image. Output only a single integer and no additional text."
    
    model = "pixtral-12b-2409"
    api_key = os.environ["PIXTRAL"]
    client = Mistral(api_key=api_key)
    img_list = os.listdir(data_folder)
    
    
    # Encode context images
    img_refs = ['2347.png', '2573.png', '4714.png']
    context_imgs = [encode_image(os.path.join('/Users/mohit/Documents/GitHub/ecdna-analysis/train_im_data/labels/', img)) for img in img_refs]
    
    for img in img_list:
        if img.endswith('.png'):    
            print(img)
            
            img_path = os.path.join(data_folder, img)
            result = generate_three_shot_response(client, model, context_imgs, img_path, prompt, context)
            ec_predictions.append(result)
            
            time.sleep(3)
        
    preds = pd.DataFrame(data={'img':img_list, 'pred':ec_predictions})
    preds['img'] = preds['img'].apply(lambda x: x[:-4])
    return preds

In [79]:
test_folder = '/Users/mohit/Documents/GitHub/ecdna-analysis/test_im_data/labels'
sampled_path = './data/sampled'
sample_df = generate_random_sample(input_folder=test_folder, output_folder=sampled_path, sample_size=100)

preds = pixtral_three_shot(sampled_path)

1637.png
610.png
189.png
2114.png
758.png
764.png
1794.png
2303.png
349.png
411.png
2329.png
773.png
1026.png
835.png
1609.png
410.png
831.png
762.png
2072.png
2265.png
1744.png
993.png
1381.png
749.png
205.png
1034.png
1790.png
774.png
1592.png
1898.png
667.png
14.png
1250.png
263.png
289.png
699.png
704.png
2149.png
1326.png
1450.png
715.png
1901.png
1732.png
1727.png
258.png
270.png
714.png
674.png
476.png
1297.png
663.png
487.png
108.png
652.png
1305.png
453.png
1879.png
2236.png
644.png
442.png
1854.png
2030.png
494.png
1869.png
333.png
657.png
1705.png
1922.png
246.png
911.png
1261.png
871.png
2337.png
56.png
989.png
1574.png
1950.png
1213.png
1415.png
803.png
55.png
791.png
2254.png
2269.png
169.png
347.png
435.png
1374.png
233.png
2251.png
391.png
385.png
1834.png
2093.png
768.png
1028.png
1766.png
1558.png
796.png
812.png


In [80]:
three_shot_df= pd.merge(preds, test_df, how='left', on='img')
three_shot_df

,img,pred,ec
0,1637,1,10
1,610,27,57
2,189,12,96
3,2114,1,4
4,758,0,0
...,...,...,...
95,1028,1,1
96,1766,1,0
97,1558,1,6
98,796,1,0


In [85]:
# Calculate the metrics
calc_metrics(three_shot_df['ec'].astype(int), three_shot_df['pred'].astype(int))

(np.float64(12.96923076923077),
 np.float64(23.43271874308032),
 np.float64(0.5789460173389579))

### Experiment #3: Multi-Layer Prompting

In [47]:
def generate_multilayer_response(client, model, input_img_path, prompt, context, temp=0.7):
    
    # Encode image
    img_input = encode_image(input_img_path)
    
    # Define the messages for the chat
    messages = [
        {
            "role":"system",
            "content": f'{context}'
            
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Describe this image."
                },
                {
                    "type": "image_url",
                    "image_url": f"data:image/jpeg;base64,{img_input}" 
                }
            ]
        },
        {
            "role": "assistant",
            "content": [
                {
                    "type": "text",
                    "text": "This is a stained image of cells containing various structures such as ecDNA, chromosomes, and nuclei."
                }
            ]
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": f"{prompt}"
                }
            ]
        }
    ]
    
    # Get the chat response
    chat_response = client.chat.complete(
        model=model,
        temperature=temp,
        messages=messages
    )

    # Return the content of the response
    return chat_response.choices[0].message.content

In [48]:
def pixtral_multi_layer(data_folder):
    ec_predictions = []
    context = """
            For this task, you will act as a pathologist who is studying 
            extrachromosomal DNA (ecDNA). You will be given multiple images
            and asked to identify the number of circular ecDNA structures.
            ecDNA is usually smaller than chromosomes or nuclei which are also present.
            """
    prompt = "Count the number of ecDNA in this image. Output only a single integer and no additional text."
    
    model = "pixtral-12b-2409"
    api_key = os.environ["PIXTRAL"]
    client = Mistral(api_key=api_key)
    img_list = os.listdir(data_folder)
    
    for img in img_list:
        if img.endswith('.png'):    
            print(img)
            
            img_path = os.path.join(data_folder, img)
            result = generate_multilayer_response(client, model, img_path, prompt, context)
            ec_predictions.append(result)
            
            time.sleep(3.5)
        
    preds = pd.DataFrame(data={'img':img_list, 'pred':ec_predictions})
    preds['img'] = preds['img'].apply(lambda x: x[:-4])
    return preds


In [49]:
test_folder = '/Users/mohit/Documents/GitHub/ecdna-analysis/test_im_data/labels'
sampled_path = './data/sampled'
sample_df = generate_random_sample(input_folder=test_folder, output_folder=sampled_path, sample_size=100)

preds = pixtral_multi_layer(sampled_path)

88.png
572.png
765.png
1346.png
2063.png
203.png
2261.png
1026.png
202.png
1796.png
438.png
2112.png
831.png
1209.png
2270.png
70.png
165.png
1630.png
761.png
985.png
211.png
2110.png
1380.png
1169.png
276.png
504.png
538.png
1325.png
328.png
1092.png
1326.png
895.png
1691.png
107.png
113.png
926.png
271.png
2170.png
476.png
845.png
514.png
1042.png
1532.png
688.png
111.png
1877.png
2340.png
646.png
929.png
2209.png
684.png
647.png
451.png
902.png
2009.png
2236.png
903.png
687.png
678.png
1868.png
668.png
293.png
1074.png
1706.png
32.png
457.png
469.png
1894.png
938.png
290.png
252.png
1329.png
1671.png
497.png
468.png
440.png
369.png
625.png
2041.png
792.png
1987.png
2054.png
2134.png
168.png
801.png
1239.png
949.png
550.png
784.png
747.png
800.png
637.png
186.png
192.png
810.png
743.png
2292.png
90.png
84.png
783.png


In [50]:
multi_layer_df= pd.merge(preds, test_df, how='left', on='img')
multi_layer_df

,img,pred,ec
0,88,5,34
1,572,3,15
2,765,1,0
3,1346,1,0
4,2063,1,0
...,...,...,...
95,743,2,4
96,2292,3,20
97,90,5,16
98,84,1,1


In [51]:
# Calculate the metrics
calc_metrics(multi_layer_df['ec'].astype(int), multi_layer_df['pred'].astype(int))

(np.float64(12.36923076923077),
 np.float64(22.363431689324358),
 np.float64(0.5792761542959773))

### Experiment #4: Temperature Adjustment

In [38]:
def generate_temp_response(client, model, img_path, prompt, context, response_format=None, temp=0.7):
    
    # Encode image
    img_input = encode_image(img_path)
    
    # Define the messages for the chat
    messages = [
        {
            "role":"system",
            "content": f'{context}'
            
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": f"{prompt}"
                },
                {
                    "type": "image_url",
                    "image_url": f"data:image/jpeg;base64,{img_input}" 
                }
            ]
        }
    ]
    
    # Get the chat response
    chat_response = client.chat.complete(
        model=model,
        temperature=temp,
        messages=messages
    )

    # Return the content of the response
    return chat_response.choices[0].message.content

In [39]:
def pixtral_temp_change(data_folder):
    ec_predictions = []
    context = """
            For this task, you will act as a pathologist who is studying 
            extrachromosomal DNA (ecDNA). You will be given multiple images
            and asked to identify the number of circular ecDNA structures.
            ecDNA is usually smaller than chromosomes or nuclei which are also present.
            """
    prompt = "Count the number of ecDNA in this image. Output only a single integer and no additional text."
    
    model = "pixtral-12b-2409"
    api_key = os.environ["PIXTRAL"]
    client = Mistral(api_key=api_key)
    img_list = os.listdir(data_folder)
    
    for img in img_list:
        if img.endswith('.png'):    
            print(img)
            
            img_path = os.path.join(data_folder, img)
            result = generate_temp_response(client, model, img_path, prompt, context, temp=0.2)
            ec_predictions.append(result)
            
            time.sleep(3.5)
        
    preds = pd.DataFrame(data={'img':img_list, 'pred':ec_predictions})
    preds['img'] = preds['img'].apply(lambda x: x[:-4])
    
    return preds

In [40]:
test_folder = '/Users/mohit/Documents/GitHub/ecdna-analysis/test_im_data/labels'
sampled_path = './data/sampled'
sample_df = generate_random_sample(input_folder=test_folder, output_folder=sampled_path, sample_size=100)

preds = pixtral_temp_change(sampled_path)

2302.png
1637.png
823.png
758.png
1569.png
1233.png
1227.png
1025.png
613.png
1595.png
565.png
2328.png
410.png
414.png
1631.png
59.png
1751.png
2265.png
1750.png
1142.png
1197.png
1829.png
198.png
826.png
1791.png
577.png
614.png
934.png
1090.png
538.png
699.png
15.png
2189.png
1133.png
1911.png
2161.png
113.png
1929.png
13.png
1109.png
662.png
892.png
925.png
514.png
1043.png
930.png
717.png
478.png
1927.png
1674.png
1460.png
916.png
1267.png
903.png
23.png
446.png
1115.png
708.png
2030.png
1074.png
331.png
119.png
1512.png
938.png
247.png
1710.png
1739.png
1261.png
1103.png
4.png
56.png
194.png
2055.png
1951.png
793.png
787.png
2040.png
801.png
829.png
791.png
236.png
544.png
545.png
237.png
1827.png
347.png
838.png
1228.png
1758.png
1349.png
1605.png
1822.png
1834.png
152.png
2093.png
225.png
782.png
1570.png
812.png
2131.png


In [45]:
temp_df= pd.merge(preds, test_df, how='left', on='img')
temp_df

,img,pred,ec
0,2302,0,0
1,1637,5,10
2,823,2,5
3,758,2,0
4,1569,15,28
...,...,...,...
95,225,15,24
96,782,1,0
97,1570,15,16
98,812,1,0


In [46]:
# Calculate the metrics
calc_metrics(temp_df['ec'].astype(int), temp_df['pred'].astype(int))

(np.float64(12.677966101694915),
 np.float64(25.29353102800016),
 np.float64(0.4148005670451608))